In [1]:
import pandas as pd

gtfs_folder = r'C:\Users\sahil\projects\boston-mbta-ridership\data\raw\MBTA GTFS' + '\\'
processed   = r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed'

In [2]:
routes = pd.read_csv(gtfs_folder + 'routes.txt', dtype={'route_id': str})

# route_type 3 = bus (0/1 = subway, 2 = commuter rail, 4 = ferry)
bus_routes = routes[routes['route_type'] == 3][['route_id', 'route_short_name', 'route_long_name']].copy()
bus_routes['route_id'] = bus_routes['route_id'].str.lstrip('0')     # '01' -> '1', to match the other files

print(len(bus_routes), 'bus routes')
bus_routes.head()

371 bus routes


,route_id,route_short_name,route_long_name
8,741,SL1,Logan Airport Terminals - South Station
9,742,SL2,Design Center - South Station
10,743,SL3,Chelsea Station - South Station
11,751,SL4,Nubian Station - South Station
12,749,SL5,Nubian Station - Temple Place


In [3]:
trips = pd.read_csv(gtfs_folder + 'trips.txt',
                    usecols=['trip_id', 'route_id', 'direction_id', 'shape_id'], dtype=str)
trips['route_id'] = trips['route_id'].str.lstrip('0')

# Keep only trips on bus routes
bus_trips = trips[trips['route_id'].isin(bus_routes['route_id'])]

print(len(bus_trips), 'bus trips')

98934 bus trips


In [4]:
# stop_times.txt is large, so load only the two columns we need
stop_times = pd.read_csv(gtfs_folder + 'stop_times.txt', usecols=['trip_id', 'stop_id'], dtype=str)

# Attach each stop visit to its route, then keep one row per route-stop pair
route_stop_pairs = (stop_times.merge(bus_trips[['trip_id', 'route_id']], on='trip_id')
                              [['route_id', 'stop_id']]
                              .drop_duplicates())

print(len(route_stop_pairs), 'route-stop pairs')

12313 route-stop pairs


In [5]:
stops = pd.read_csv(gtfs_folder + 'stops.txt',
                    usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'], dtype={'stop_id': str})

route_stops = route_stop_pairs.merge(stops, on='stop_id', how='left')
route_stops['fare_free'] = route_stops['route_id'].isin(['23', '28', '29'])

print(route_stops.shape)
print('Stops missing a location:', route_stops['stop_lat'].isna().sum())     # expect 0
route_stops[route_stops['route_id'] == '28'].head()

(12313, 6)
Stops missing a location: 0


,route_id,stop_id,stop_name,stop_lat,stop_lon,fare_free
2106,28,390,Warren St @ Quincy St,42.316257,-71.082891,True
2107,28,392,Warren St @ Maywood St,42.318157,-71.081788,True
2108,28,393,Warren St @ Woodbine St,42.319956,-71.081380,True
2109,28,394,Warren St @ Waverly St,42.321585,-71.081813,True
2110,28,395,Warren St @ Whiting St,42.323909,-71.082602,True


In [6]:
stops = pd.read_csv(gtfs_folder + 'stops.txt',
                    usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'], dtype={'stop_id': str})

route_stops = route_stop_pairs.merge(stops, on='stop_id', how='left')
route_stops['fare_free'] = route_stops['route_id'].isin(['23', '28', '29'])

print(route_stops.shape)
print('Stops missing a location:', route_stops['stop_lat'].isna().sum())     # expect 0
route_stops[route_stops['route_id'] == '28'].head()

(12313, 6)
Stops missing a location: 0


,route_id,stop_id,stop_name,stop_lat,stop_lon,fare_free
2106,28,390,Warren St @ Quincy St,42.316257,-71.082891,True
2107,28,392,Warren St @ Maywood St,42.318157,-71.081788,True
2108,28,393,Warren St @ Woodbine St,42.319956,-71.081380,True
2109,28,394,Warren St @ Waverly St,42.321585,-71.081813,True
2110,28,395,Warren St @ Whiting St,42.323909,-71.082602,True


In [7]:
route_stops.to_csv(processed + r'\route_stops.csv', index=False)
print('Saved route_stops.csv')

Saved route_stops.csv


In [8]:
ridership_stops = pd.read_csv(processed + r'\ridership_stops_year.csv', dtype={'route_id': str, 'stop_id': str})

latest = ridership_stops[ridership_stops['year'] == 2025][['route_id', 'stop_id']].drop_duplicates()
matched = latest.merge(route_stops[['route_id', 'stop_id']], on=['route_id', 'stop_id'],
                       how='left', indicator=True)

print(matched['_merge'].value_counts())

_merge
both          9968
left_only     1147
right_only       0
Name: count, dtype: int64


In [9]:
unmatched = matched[matched['_merge'] == 'left_only']

# Routes with the most unmatched stops
print(unmatched['route_id'].value_counts().head(10))

# Fare-free routes specifically
print(unmatched[unmatched['route_id'].isin(['23', '28', '29'])]['route_id'].value_counts())

route_id
103    166
191    131
193     87
192     82
66      44
33      40
194     39
747     38
435     37
21      32
Name: count, dtype: int64
route_id
23    1
28    1
Name: count, dtype: int64


In [11]:
patterns = pd.read_csv(gtfs_folder + 'route_patterns.txt', dtype=str)
patterns['route_id'] = patterns['route_id'].str.lstrip('0')

# Typicality 1 = the route's main, everyday pattern
main_patterns = patterns[(patterns['route_pattern_typicality'] == '1') &
                         (patterns['route_id'].isin(bus_routes['route_id']))]

# Each pattern has a representative trip; that trip tells us the shape
main_shapes = main_patterns.merge(trips[['trip_id', 'shape_id']],
                                  left_on='representative_trip_id', right_on='trip_id')
main_shapes = main_shapes[['route_id', 'direction_id', 'shape_id']].drop_duplicates()

print(len(main_shapes), 'main shapes')
main_shapes.head()

334 main shapes


,route_id,direction_id,shape_id
0,741,0,7410062
1,741,1,7410061
2,742,0,7420075
3,742,1,7420074
4,743,0,7430059


In [12]:
shapes = pd.read_csv(gtfs_folder + 'shapes.txt', dtype={'shape_id': str})

# Attach the points (latitude/longitude, in order) for each main shape
route_shapes = (main_shapes.merge(shapes, on='shape_id')
                           .sort_values(['route_id', 'direction_id', 'shape_pt_sequence']))
route_shapes['fare_free'] = route_shapes['route_id'].isin(['23', '28', '29'])

print(route_shapes.shape)
print('Routes with a shape:', route_shapes['route_id'].nunique())

route_shapes.to_csv(processed + r'\route_shapes.csv', index=False)
print('Saved route_shapes.csv')

(108432, 8)
Routes with a shape: 152
Saved route_shapes.csv


In [13]:
top8_ids = ['1', '66', '39', '57', '111', '32', '22', '9']

check_routes = ['23', '28', '29'] + top8_ids
print([r for r in check_routes if r not in route_shapes['route_id'].unique()])

[]
